In [1]:
import numpy as np
import pandas as pd
import random
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score, brier_score_loss, fbeta_score, make_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

from gensim.models import Word2Vec

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from collections import Counter
import time

import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

torch.backends.cudnn.enabled = False
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Unai\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Unai\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
c:\Users\Unai\miniconda3\envs\nlp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%load_ext autoreload
%autoreload 2

from models import lstm, pretrained_transformer
from train import train_inference

In [3]:
df = pd.read_csv("../data/reduced_dataset.csv")

print(df['clase'].value_counts())

X = df['texto']
y = df['clase'] == 'susp'

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

clase
susp    1929
src     1891
Name: count, dtype: int64


In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [6]:
def c_at_1(y_true, y_prob, threshold=0.5, uncertain_band=0.05):
    y_pred = np.full_like(y_true, -1)

    uncertain = np.abs(y_prob - threshold) <= uncertain_band
    y_pred[~uncertain] = (y_prob[~uncertain] >= threshold).astype(int)

    n = len(y_true)
    n_correct = np.sum(y_pred == y_true)
    n_unanswered = np.sum(uncertain)

    return (1/n) * (n_correct + n_unanswered * (n_correct/n))

In [7]:
def composite_metric(y_true, y_prob, **kwargs):
    y_prob = np.array(y_prob, dtype=float)

    if y_prob.ndim == 2:
        y_prob = y_prob[:, 1]

    y_pred = (y_prob >= 0.5).astype(int)

    roc = roc_auc_score(y_true, y_prob)
    brier = 1 - brier_score_loss(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    f05u = fbeta_score(y_true, y_pred, beta=0.5)
    c1 = c_at_1(y_true, y_prob)

    return np.mean([roc, brier, f1, f05u, c1])


custom_scorer = make_scorer(
    composite_metric,
    needs_proba=True,
    greater_is_better=True
)

### Experiment 1

In [8]:
def build_vocab(texts, min_freq=30):
    counter = Counter()
    for text in texts:
        counter.update(text.split())

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab = build_vocab(X_train_clean)

In [9]:
from sklearn.model_selection import train_test_split

lstm_train_texts, lstm_val_texts, lstm_train_labels, lstm_val_labels = train_test_split(
    X_train_clean,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=SEED
)

In [10]:
lstm_train_dataset = lstm.LSTMDataset(lstm_train_texts, lstm_train_labels, vocab)
lstm_val_dataset = lstm.LSTMDataset(lstm_val_texts, lstm_val_labels, vocab)

train_loader = DataLoader(
    lstm_train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=lstm.collate_lstm
)

val_loader = DataLoader(
    lstm_val_dataset,
    batch_size=8,
    collate_fn=lstm.collate_lstm
)

In [11]:
def count_parameters(model):
    return sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

In [12]:
model = lstm.LSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_dim=128,
    output_dim=2
)

model, val_score, training_time = train_inference.train_model(
    model,
    train_loader,
    val_loader,
    lr=1e-3,
    metric = composite_metric
)

In [13]:
test_dataset = lstm.LSTMDataset(X_test_clean, y_test, vocab)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    collate_fn=lstm.collate_lstm
)

start = time.time()
test_metric = train_inference.evaluate_model(model, test_loader, composite_metric)
inference_time = time.time() - start
params = count_parameters(model)
gpu_memory = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0

In [14]:
print(f"Train Score: {val_score}")
print(f"Test Score: {test_metric}")
print(f"Training time: {training_time}")
print(f"Inference time: {inference_time}")
print(f"Model params: {params}")
print(f"GPU Memory: {gpu_memory}")

Train Score: 0.6732676225466437
Test Score: 0.6468036004833679
Training time: 829.357164144516
Inference time: 6.380653381347656
Model params: 2589434
GPU Memory: 83822080


In [15]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [16]:
bert_train_dataset = pretrained_transformer.BERTDataset(X_train, y_train, tokenizer)
bert_test_dataset = pretrained_transformer.BERTDataset(X_test, y_test, tokenizer)

In [17]:
train_loader = DataLoader(
    bert_train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = train_loader
test_loader = DataLoader(
    bert_test_dataset,
    batch_size=8
)

In [18]:
model = pretrained_transformer.TransformerClassifier(
    model_name="bert-base-uncased",
    output_dim=2,
    pooling="cls",
    freeze_encoder=False
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 711.08it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
model, val_score, training_time = train_inference.train_model(
    model,
    train_loader,
    val_loader,
    metric = composite_metric,
    lr=2e-5,
    scheduler=True
)

In [20]:
start = time.time()

test_metric = train_inference.evaluate_model(model, test_loader, composite_metric)

inference_time = time.time() - start

params = count_parameters(model)

gpu_memory = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0

In [21]:
print(f"Train Score: {val_score}")
print(f"Test Score: {test_metric}")
print(f"Training time: {training_time}")
print(f"Inference time: {inference_time}")
print(f"Model params: {params}")
print(f"GPU Memory: {gpu_memory}")

Train Score: 0.999391093896115
Test Score: 0.7005407771131276
Training time: 2762.844085454941
Inference time: 27.306763648986816
Model params: 109483778
GPU Memory: 2642249728


In [24]:
tokenized = [text.split() for text in X_train_clean]

w2v_model = Word2Vec(
    sentences=tokenized,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=SEED
)


In [25]:
class W2VVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, model):
        self.model = model
        self.dim = model.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        vectors = []
        for text in X:
            words = text.split()
            word_vecs = [self.model.wv[w] for w in words if w in self.model.wv]
            if len(word_vecs) == 0:
                vectors.append(np.zeros(self.dim))
            else:
                vectors.append(np.mean(word_vecs, axis=0))
        return np.array(vectors)


In [26]:
w2v_vectorizer = W2VVectorizer(w2v_model)

X_train_w2v = w2v_vectorizer.transform(X_train_clean)
X_test_w2v = w2v_vectorizer.transform(X_test_clean)

clf = LogisticRegression(max_iter=1000, random_state=SEED)

param_grid = {
    "C": [0.01, 0.1, 1, 10]
}

grid_w2v = GridSearchCV(
    clf,
    param_grid,
    cv=skf,
    scoring=custom_scorer,
    n_jobs=-1
)

grid_w2v.fit(X_train_w2v, y_train)

print("Best W2V Score:", grid_w2v.best_score_)


Best W2V Score: 0.8814510291422393


In [31]:
grid_w2v.best_params_

{'C': 0.1}

In [30]:
best_w2v = grid_w2v.best_estimator_
y_pred_w2v = best_w2v.predict(X_test_w2v)

print("Dense Test Score:",
      composite_metric(y_test, y_pred_w2v, average="macro"))

Dense Test Score: 0.8977402796093727


### Experiment 2

In [37]:
fractions = [0.25, 0.50, 0.75, 1.0]
lc_results = {
    "BASE": [],
    "LSTM": [],
    "BERT": []
}

In [ ]:
for frac in fractions:

    X_sub = X_train_clean.sample(
        frac=frac,
        random_state=SEED
    )

    y_sub = y_train.loc[X_sub.index]

    #BASELINE
    tokenized = [text.split() for text in X_sub]

    w2v_model = Word2Vec(
        sentences=tokenized,
        vector_size=100,
        window=5,
        min_count=2,
        workers=4,
        seed=SEED
    )
    X_train_w2v = w2v_vectorizer.transform(X_sub)
    X_test_w2v = w2v_vectorizer.transform(X_test_clean)
    clf = LogisticRegression(max_iter=1000, random_state=SEED, C=0.1)
    clf.fit(X_train_w2v, y_sub)
    y_pred = clf.predict(X_test_w2v)
    lc_results["BASE"].append(
        composite_metric(y_test, y_pred, average="macro")
    )



    # LSTM
    dataset = lstm.LSTMDataset(X_sub, y_sub, vocab)

    loader = DataLoader(
        dataset,
        batch_size=8,
        shuffle=True,
        collate_fn=lstm.collate_lstm
    )

    model = lstm.LSTMClassifier(
        vocab_size=len(vocab),
        embedding_dim=100,
        hidden_dim=128,
        output_dim=2
    )

    model, _, _ = train_inference.train_model(
        model,
        loader,
        loader,
        metric = composite_metric,
        lr=1e-3
    )

    test_dataset = lstm.LSTMDataset(X_test_clean, y_test, vocab)
    test_loader = DataLoader(
        test_dataset,
        batch_size=8,
        collate_fn=lstm.collate_lstm
    )

    lc_results["LSTM"].append(
        train_inference.evaluate_model(model, test_loader, metric = composite_metric,)
    )

    #BERT
    dataset = pretrained_transformer.BERTDataset(X_sub, y_sub, tokenizer)

    loader = DataLoader(
        dataset,
        batch_size=8,
        shuffle=True,
    )

    model = pretrained_transformer.TransformerClassifier(
        model_name="bert-base-uncased",
        output_dim=2,
        pooling="cls",
        freeze_encoder=False
    )

    model, _, _ = train_inference.train_model(
        model,
        loader,
        loader,
        metric = composite_metric,
        lr=1e-3
    )

    test_dataset = pretrained_transformer.BERTDataset(X_test_clean, y_test, tokenizer)
    test_loader = DataLoader(
        test_dataset,
        batch_size=8,
    )

    lc_results["BERT"].append(
        train_inference.evaluate_model(model, test_loader, metric = composite_metric,)
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 670.27it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 635.17it/s, Materializing param=pooler.dense.weight]                           

In [ ]:
plt.plot([25,50,75,100], lc_results["LSTM"], label="LSTM")
plt.xlabel("Training Data (%)")
plt.ylabel("Metric")
plt.legend()
plt.show()